## Чтение файла CSV со штрафами

In [55]:
import pandas as pd
import gc

df = pd.read_csv("data/fines.csv")
df.head()

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.0,Ford,Focus,1989
1,E432XX77RUS,1,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1,2100.0,Ford,Focus,1984
3,X582HE161RUS,2,2000.0,Ford,Focus,2015
4,92918M178RUS,1,5700.0,Ford,Focus,2014


## Итерации - использование цикла for с iloc

In [56]:
%%timeit
def calc_l(df):
    res = []
    for i in range(0, len(df)):
        val = df.iloc[i]['Fines'] / df.iloc[i]['Refund'] * df.iloc[i]['Year']
        res.append(val)
    return res

df['calc_loop'] = calc_l(df)

45 ms ± 878 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Итерации - использование iterrows

In [57]:
%%timeit
def calc_iterrows(df):
    res = []
    for _, row in df.iterrows():
        res.append(row['Fines'] / row['Refund'] * row['Year'])
    return res

df['calc_iterrows'] = calc_iterrows(df)

11.7 ms ± 119 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Итерации - использование apply с lambda

In [58]:
%%timeit
df['calc_apply'] = df.apply(lambda row: row['Fines'] / row['Refund'] * row['Year'], axis=1)

3.61 ms ± 142 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Итерации - использование объектов Series

In [59]:
%%timeit
df['calc_series'] = df['Fines'] / df['Refund'] * df['Year']

56.5 μs ± 685 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Итерации - использование .values

In [60]:
%%timeit
df['calc_values'] = (df['Fines'].values / df['Refund'].values) * df['Year'].values

28.9 μs ± 173 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Индексирование - получение строки без индекса

In [61]:
%%timeit
df[df['CarNumber'] == 'O136HO197RUS']

172 μs ± 1.56 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Индексирование - установка CarNumber в качестве индекса и получение строки по новому индексу

In [62]:
df = df.set_index('CarNumber')

In [63]:
%%timeit
df.loc['O136HO197RUS']

24.2 μs ± 423 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Понижающее преобразование (даункастинг) - оптимизация float и integer

In [64]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to BAB064OR940RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    int64  
 1   Fines          930 non-null    float64
 2   Make           930 non-null    str    
 3   Model          919 non-null    str    
 4   Year           930 non-null    int64  
 5   calc_loop      930 non-null    float64
 6   calc_iterrows  930 non-null    float64
 7   calc_apply     930 non-null    float64
 8   calc_series    930 non-null    float64
 9   calc_values    930 non-null    float64
dtypes: float64(6), int64(2), str(2)
memory usage: 243.4 KB


In [ ]:
optimized_df = df.copy()

float_cols = optimized_df.select_dtypes('float').columns.tolist()
int_cols = optimized_df.select_dtypes('int').columns.tolist()

for col in float_cols:
    optimized_df[col] = optimized_df[col].astype('float32')

for col in int_cols:
    col_min = optimized_df[col].min()
    col_max = optimized_df[col].max()
    if col_min >= -128 and col_max <= 127:
        optimized_df[col] = optimized_df[col].astype('int8')
    elif col_min >= -32768 and col_max <= 32767:
        optimized_df[col] = optimized_df[col].astype('int16')
    elif col_min >= -2147483648 and col_max <= 2147483647:
        optimized_df[col] = optimized_df[col].astype('int32')

optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to BAB064OR940RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    int8   
 1   Fines          930 non-null    float32
 2   Make           930 non-null    str    
 3   Model          919 non-null    str    
 4   Year           930 non-null    int16  
 5   calc_loop      930 non-null    float32
 6   calc_iterrows  930 non-null    float32
 7   calc_apply     930 non-null    float32
 8   calc_series    930 non-null    float32
 9   calc_values    930 non-null    float32
dtypes: float32(6), int16(1), int8(1), str(2)
memory usage: 209.8 KB


## Категории - преобразование нужных столбцов в тип category

In [66]:
obj_cols = optimized_df.select_dtypes('object').columns
optimized_df[obj_cols] = optimized_df[obj_cols].astype('category')
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to BAB064OR940RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Refund         930 non-null    int8    
 1   Fines          930 non-null    float32 
 2   Make           930 non-null    category
 3   Model          919 non-null    category
 4   Year           930 non-null    int16   
 5   calc_loop      930 non-null    float32 
 6   calc_iterrows  930 non-null    float32 
 7   calc_apply     930 non-null    float32 
 8   calc_series    930 non-null    float32 
 9   calc_values    930 non-null    float32 
dtypes: category(2), float32(6), int16(1), int8(1)
memory usage: 115.1 KB


/tmp/ipykernel_40509/2062800837.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = optimized_df.select_dtypes('object').columns


## Очистка памяти - удаление исходного dataframe

In [67]:
%reset_selective -f df
gc.collect()

0